# ApexID Split V2 dataset analysis

This notebook summarizes the observation-aware partitioning described in Sec. 2.3.

It reports:
- frozen ApexID Split V2 train and validation counts;
- five deterministic external test splits for every benchmark dataset;
- identities, images, and observation groups, where an observation is one `observation_group_id`;
- a headline dataset/species table and a `Total Multi-species` row;
- integrity checks for Split V2, identity leakage, image overlap, and observation leakage.

The notebook reads existing artifacts only. It does not rebuild Split V2 or regenerate test partitions.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 48)

## 1. Resolve the frozen benchmark artifacts

The candidate paths mirror notebooks 00 and 03. Set an environment variable only when the attached prepared bundle lives elsewhere.

In [ ]:
HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE

PREPARED_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_PREPARED_INPUT", REPO_ROOT / "outputs" / "apexid_cvpr_prepared")),
    REPO_ROOT / "outputs" / "apexid_cvpr_prepared",
)
PREBUILT_SPLIT_ROOT_CANDIDATES = (
    Path(os.environ.get("APEXID_SPLIT_ROOT", REPO_ROOT / "open_set_splits_v2")),
)
CORE_SPLIT_V2_FILENAMES = ("train.csv", "val_known.csv", "val_unknown.csv", "val.csv")
OPTIONAL_SPLIT_V2_FILENAMES = ("val_unknown_extra.csv", "unlabeled_ood.csv")
SPLIT_V2_COMPONENT_FILENAMES = CORE_SPLIT_V2_FILENAMES


def first_existing(candidates: tuple[Path, ...], marker: str) -> Path:
    for candidate in candidates:
        if str(candidate) and (candidate / marker).is_file():
            return candidate.resolve()
    checked = [str(candidate) for candidate in candidates if str(candidate)]
    raise FileNotFoundError(f"Could not find {marker}. Checked: {checked}")


def has_complete_split_v2(root: Path) -> bool:
    return all((root / filename).is_file() for filename in (*SPLIT_V2_COMPONENT_FILENAMES, "split_manifest.csv"))

PREPARED_ROOT = first_existing(PREPARED_ROOT_CANDIDATES, "preparation_manifest.json")
PREPARED_SPLIT_ROOT = PREPARED_ROOT / "apexid_split_v2"
PREBUILT_SPLIT_ROOT = next(
    (
        candidate.resolve()
        for candidate in PREBUILT_SPLIT_ROOT_CANDIDATES
        if str(candidate) and has_complete_split_v2(candidate)
    ),
    None,
)
if has_complete_split_v2(PREPARED_SPLIT_ROOT):
    SPLIT_V2_ROOT = PREPARED_SPLIT_ROOT
elif PREBUILT_SPLIT_ROOT is not None:
    SPLIT_V2_ROOT = PREBUILT_SPLIT_ROOT
else:
    raise FileNotFoundError("Set APEXID_SPLIT_ROOT to a complete frozen Split V2 bundle with split_manifest.csv.")

PREPARATION_MANIFEST_PATH = PREPARED_ROOT / "preparation_manifest.json"
TEST_SPLITS_ROOT = PREPARED_ROOT / "splits"

PREPARATION_MANIFEST = json.loads(PREPARATION_MANIFEST_PATH.read_text(encoding="utf-8"))
if PREPARATION_MANIFEST.get("status") != "passed":
    raise RuntimeError("preparation_manifest.json is not marked passed")
if not PREPARATION_MANIFEST.get("apexid_split_v2", {}).get("passed"):
    raise RuntimeError("The prepared ApexID Split V2 audit is not marked passed")

EXTERNAL_DATASETS = ("bristol_gorillas", "elephants", "gcn_id", "jaguar_antonio")
SPLIT_SEEDS = tuple(PREPARATION_MANIFEST.get("protocol", {}).get("split_seeds", (42, 123, 3407, 2026, 1337)))

print("Prepared root:", PREPARED_ROOT)
print("Split V2 root:", SPLIT_V2_ROOT)
print("External test root:", TEST_SPLITS_ROOT)
print("Datasets:", EXTERNAL_DATASETS)
print("Test seeds:", SPLIT_SEEDS)

## 2. Load and validate ApexID Split V2

`split_manifest.csv` is the authoritative source for split methods and observation groups. The public CSVs provide the train, validation-known, and validation-unknown membership.

In [ ]:
SPLIT_V2_FILES = {
    "train": SPLIT_V2_ROOT / "train.csv",
    "val_known": SPLIT_V2_ROOT / "val_known.csv",
    "val_unknown": SPLIT_V2_ROOT / "val_unknown.csv",
    "val": SPLIT_V2_ROOT / "val.csv",
    "val_unknown_extra": SPLIT_V2_ROOT / "val_unknown_extra.csv",
    "unlabeled_ood": SPLIT_V2_ROOT / "unlabeled_ood.csv",
    "manifest": SPLIT_V2_ROOT / "split_manifest.csv",
}
CORE_SPLIT_NAMES = ("train", "val_known", "val_unknown")
OPTIONAL_SPLIT_NAMES = ("val_unknown_extra", "unlabeled_ood")
AVAILABLE_COMPONENT_SPLIT_NAMES = CORE_SPLIT_NAMES + tuple(
    name for name in OPTIONAL_SPLIT_NAMES if SPLIT_V2_FILES[name].is_file()
)
required_public_files = [SPLIT_V2_FILES[name] for name in CORE_SPLIT_NAMES] + [SPLIT_V2_FILES["val"]]
missing = [str(path) for path in required_public_files if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing core public Split V2 artifacts: {missing}")

public_splits = {
    name: pd.read_csv(path, dtype=str).fillna("")
    for name, path in SPLIT_V2_FILES.items()
    if name in set(AVAILABLE_COMPONENT_SPLIT_NAMES) | {"val"}
}

required_public_columns = {"path", "identity", "dataset", "species"}
for split_name, frame in public_splits.items():
    if not required_public_columns.issubset(frame.columns):
        raise ValueError(f"Split V2 {split_name} lacks {sorted(required_public_columns - set(frame.columns))}")

if SPLIT_V2_FILES["manifest"].is_file():
    split_manifest = pd.read_csv(SPLIT_V2_FILES["manifest"], dtype=str).fillna("")
    required_manifest_columns = {
        "path", "identity", "dataset", "species", "canonical_identity",
        "split", "observation_group_id", "image_sha256",
    }
    if not required_manifest_columns.issubset(split_manifest.columns):
        raise ValueError(f"Split V2 manifest lacks {sorted(required_manifest_columns - set(split_manifest.columns))}")
    observation_source = "Split V2 split_manifest.csv"
else:
    import hashlib

    parts = []
    for split_name in AVAILABLE_COMPONENT_SPLIT_NAMES:
        part = public_splits[split_name].copy()
        part["split"] = split_name
        parts.append(part)
    split_manifest = pd.concat(parts, ignore_index=True)
    split_manifest["canonical_identity"] = (
        split_manifest["dataset"].astype(str) + "::" + split_manifest["identity"].astype(str)
    )
    split_manifest["image_sha256"] = split_manifest["path"].astype(str).map(
        lambda value: hashlib.sha256(value.encode("utf-8")).hexdigest()
    )
    split_manifest["observation_group_id"] = split_manifest["path"].astype(str).map(
        lambda value: "path_observation::" + hashlib.sha1(value.encode("utf-8")).hexdigest()[:16]
    )
    observation_source = "role-only CSV fallback; observation count is unavailable"


def identity_keys(frame: pd.DataFrame) -> set[tuple[str, str]]:
    return set(zip(frame["dataset"].astype(str), frame["identity"].astype(str)))

train_ids = identity_keys(public_splits["train"])
known_ids = identity_keys(public_splits["val_known"])
unknown_ids = identity_keys(public_splits["val_unknown"])

if not known_ids.issubset(train_ids):
    raise RuntimeError(f"Split V2 val_known identities absent from train: {sorted(known_ids - train_ids)[:5]}")
if train_ids & unknown_ids:
    raise RuntimeError(f"Split V2 train/val_unknown identity overlap: {sorted(train_ids & unknown_ids)[:5]}")
if known_ids & unknown_ids:
    raise RuntimeError(f"Split V2 val_known/val_unknown identity overlap: {sorted(known_ids & unknown_ids)[:5]}")

manifest_by_path = split_manifest.set_index("path", drop=False)
for split_name, frame in public_splits.items():
    if split_name == "val":
        continue
    if frame["path"].duplicated().any():
        raise RuntimeError(f"Duplicate paths inside Split V2 {split_name}")
    absent = sorted(set(frame["path"]) - set(manifest_by_path.index))
    if absent:
        raise RuntimeError(f"Split V2 {split_name} paths absent from split analysis manifest: {absent[:5]}")

split_manifest["split"] = split_manifest["split"].astype(str)
crossing_groups = (
    split_manifest.loc[split_manifest["split"].isin(["train", "val_known"])]
    .groupby("observation_group_id")["split"]
    .nunique()
)
if (crossing_groups > 1).any():
    raise RuntimeError("Split V2 observation groups cross train and val_known")

full_dataset_inventory = (
    pd.concat([public_splits[name] for name in AVAILABLE_COMPONENT_SPLIT_NAMES], ignore_index=True)
    .groupby("dataset", as_index=False)
    .agg(images=("path", "nunique"), identities=("identity", "nunique"), species=("species", "first"))
    .sort_values("dataset")
    .reset_index(drop=True)
)
sea_turtle_datasets = {"AmvrakikosTurtles", "ReunionTurtles", "ZakynthosTurtles"}
missing_sea_turtle_datasets = sea_turtle_datasets - set(full_dataset_inventory["dataset"])
if missing_sea_turtle_datasets:
    raise RuntimeError(f"Missing expected sea-turtle datasets: {sorted(missing_sea_turtle_datasets)}")
if not (full_dataset_inventory["species"].astype(str).str.lower() == "sea turtle").any():
    raise RuntimeError("No sea-turtle rows were found in the complete Split V2 inventory")

print("Split V2 validation: PASSED")
print("Observation metadata:", observation_source)
print("Optional component files available:", sorted(set(AVAILABLE_COMPONENT_SPLIT_NAMES) - set(CORE_SPLIT_NAMES)))
print("Train identities:", len(train_ids), "known validation identities:", len(known_ids), "unknown validation identities:", len(unknown_ids))
print("Complete Split V2 dataset inventory:")
display(full_dataset_inventory)
print("Sea Turtle 2022 naming: source rows are represented by AmvrakikosTurtles, ReunionTurtles, and ZakynthosTurtles; no separate seaturtle2022 label exists in the authoritative catalog.")

## 3. Build the train/validation analysis table

Observation counts come from unique non-empty `observation_group_id` values. Identity counts use `(dataset, identity)` keys so reused labels across datasets cannot merge.

In [ ]:
def add_split_role(frame: pd.DataFrame, role: str) -> pd.DataFrame:
    result = frame.copy()
    result["analysis_role"] = role
    return result

train_manifest = split_manifest.loc[split_manifest["split"].eq("train")].copy()
val_known_manifest = split_manifest.loc[split_manifest["split"].eq("val_known")].copy()
val_unknown_manifest = split_manifest.loc[split_manifest["split"].eq("val_unknown")].copy()

# Keep validation visible as one partition and retain known/unknown detail separately.
val_manifest = pd.concat([
    add_split_role(val_known_manifest, "validation_known"),
    add_split_role(val_unknown_manifest, "validation_unknown"),
], ignore_index=True)

def count_partition(frame: pd.DataFrame, role: str) -> dict[str, object]:
    identities = set(zip(frame["dataset"].astype(str), frame["identity"].astype(str)))
    groups = frame.loc[frame["observation_group_id"].astype(str).ne(""), "observation_group_id"].nunique()
    species = sorted(set(frame["species"].astype(str)) - {"", "nan"})
    return {
        "partition": role,
        "images": int(len(frame)),
        "identities": int(len(identities)),
        "observations": int(groups),
        "species_taxa": "; ".join(species) if species else "unknown",
    }

train_validation_summary = pd.DataFrame([
    count_partition(train_manifest, "train"),
    count_partition(val_manifest, "validation"),
    count_partition(val_known_manifest, "validation_known"),
    count_partition(val_unknown_manifest, "validation_unknown"),
])
display(train_validation_summary)

## 4. Load all five observation-aware external test splits

Each dataset/seed manifest is the test set for that deterministic benchmark replicate. The main table below uses the mean test size across seeds; the next table preserves every seed exactly.

In [ ]:
test_manifests: dict[tuple[str, int], pd.DataFrame] = {}
test_records: list[dict[str, object]] = []

def validate_external_test(frame: pd.DataFrame, dataset: str, seed: int) -> None:
    required = {"image_id", "identity", "dataset", "species", "observation_group_id", "image_sha256", "closed_role"}
    if not required.issubset(frame.columns):
        raise ValueError(f"{dataset}/seed_{seed} missing {sorted(required - set(frame.columns))}")
    if frame["image_id"].duplicated().any():
        raise RuntimeError(f"{dataset}/seed_{seed} contains duplicate image_id values")
    if set(frame["dataset"].astype(str)) != {dataset}:
        raise RuntimeError(f"{dataset}/seed_{seed} has mixed dataset labels")

    role_values = set(frame["closed_role"])
    if not {"gallery", "query_known"}.issubset(role_values):
        raise RuntimeError(f"{dataset}/seed_{seed} has incomplete closed-set roles: {role_values}")
    gallery_ids = set(frame.loc[frame["closed_role"].eq("gallery"), "identity"].astype(str))
    known_ids = set(frame.loc[frame["closed_role"].eq("query_known"), "identity"].astype(str))
    if not known_ids.issubset(gallery_ids):
        raise RuntimeError(f"{dataset}/seed_{seed} has query identities absent from gallery")
    active = frame.loc[frame["closed_role"].ne("excluded")].copy()
    group_role_counts = active.groupby("observation_group_id")["closed_role"].nunique()
    if (group_role_counts > 1).any():
        raise RuntimeError(f"{dataset}/seed_{seed} observation group crosses test roles")

for dataset in EXTERNAL_DATASETS:
    for seed in SPLIT_SEEDS:
        path = TEST_SPLITS_ROOT / dataset / f"seed_{seed}" / "split_manifest.csv"
        if not path.is_file():
            raise FileNotFoundError(f"Missing external test split: {path}")
        frame = pd.read_csv(path, dtype=str).fillna("")
        validate_external_test(frame, dataset, int(seed))
        frame = frame.loc[frame["closed_role"].isin(["gallery", "query_known"])].copy()
        test_manifests[(dataset, int(seed))] = frame
        test_records.append({
            "dataset": dataset,
            "seed": int(seed),
            "images": int(len(frame)),
            "identities": int(frame["identity"].nunique()),
            "observations": int(frame.loc[frame["observation_group_id"].ne(""), "observation_group_id"].nunique()),
            "gallery_images": int((frame["closed_role"] == "gallery").sum()),
            "query_known_images": int((frame["closed_role"] == "query_known").sum()),
            "gallery_identities": int(frame.loc[frame["closed_role"] == "gallery", "identity"].nunique()),
            "query_known_identities": int(frame.loc[frame["closed_role"] == "query_known", "identity"].nunique()),
        })

test_by_seed = pd.DataFrame(test_records).sort_values(["dataset", "seed"]).reset_index(drop=True)
display(test_by_seed)

## 5. Headline table: Dataset / Species-Taxon / Identities / Images / Observations / Train / Validation / Test

The `Test` columns are the mean counts over the five frozen seeds. Test seed variability is shown separately so no replicate is hidden. `Train` and `Validation` refer to ApexID Split V2; external datasets have dashes for those columns because they are evaluation-only.

In [ ]:
def species_text(frame: pd.DataFrame) -> str:
    values = sorted(set(frame["species"].astype(str)) - {"", "nan"})
    return "; ".join(values) if values else "unknown"

headline_rows: list[dict[str, object]] = []
headline_datasets = set(split_manifest["dataset"].astype(str)) | set(EXTERNAL_DATASETS)
for dataset in sorted(headline_datasets):
    train = train_manifest.loc[train_manifest["dataset"].astype(str).eq(dataset)]
    validation = val_manifest.loc[val_manifest["dataset"].astype(str).eq(dataset)]
    dataset_tests = [test_manifests[(dataset, seed)] for seed in SPLIT_SEEDS if (dataset, seed) in test_manifests]
    if train.empty and validation.empty and not dataset_tests:
        continue
    test_images = [len(frame) for frame in dataset_tests]
    test_identities = [frame["identity"].nunique() for frame in dataset_tests]
    test_observations = [frame.loc[frame["observation_group_id"].ne(""), "observation_group_id"].nunique() for frame in dataset_tests]
    source = train if not train.empty else (validation if not validation.empty else dataset_tests[0])
    headline_rows.append({
        "Dataset": dataset,
        "Species / Taxon": species_text(source),
        "# Identities": int(source["identity"].nunique()),
        "# Images": int(len(source)),
        "# Observations": int(source.loc[source["observation_group_id"].ne(""), "observation_group_id"].nunique()),
        "Train Images": int(len(train)) if not train.empty else np.nan,
        "Validation Images": int(len(validation)) if not validation.empty else np.nan,
        "Test Images (5-seed mean)": round(float(np.mean(test_images)), 1) if test_images else np.nan,
        "Test Identities (5-seed mean)": round(float(np.mean(test_identities)), 1) if test_identities else np.nan,
        "Test Observations (5-seed mean)": round(float(np.mean(test_observations)), 1) if test_observations else np.nan,
    })

train_all = train_manifest
validation_all = val_manifest
multi_species = pd.concat([
    train_all.assign(_analysis_source="train"),
    validation_all.assign(_analysis_source="validation"),
], ignore_index=True)
multi_identity_count = len(set(zip(multi_species["dataset"].astype(str), multi_species["identity"].astype(str))))
multi_observation_count = multi_species.loc[multi_species["observation_group_id"].ne(""), "observation_group_id"].nunique()
headline_rows.append({
    "Dataset": "Total Multi-species",
    "Species / Taxon": "; ".join(sorted(set(multi_species["species"].astype(str)) - {"", "nan"})),
    "# Identities": int(multi_identity_count),
    "# Images": int(len(multi_species)),
    "# Observations": int(multi_observation_count),
    "Train Images": int(len(train_all)),
    "Validation Images": int(len(validation_all)),
    "Test Images (5-seed mean)": round(float(test_by_seed["images"].sum() / len(SPLIT_SEEDS)), 1),
    "Test Identities (5-seed mean)": round(float(test_by_seed.groupby("dataset")["identities"].mean().sum()), 1),
    "Test Observations (5-seed mean)": round(float(test_by_seed.groupby("dataset")["observations"].mean().sum()), 1),
})

headline_table = pd.DataFrame(headline_rows)
display(headline_table)

In [ ]:
# Paper-format corpus table: train IDs, train+validation images, and observation groups.

PAPER_DATASET_ORDER = (
    "AmvrakikosTurtles", "BalearicLizard", "CTai", "CzechLynx", "GiraffeZebraID",
    "Giraffes", "HappyWhale", "HyenaID2022", "IPanda50", "Jaguar_data",
    "LeopardID2022", "NDD20", "NyalaData", "ReunionTurtles", "SalamanderID2025",
    "SealID", "WhaleSharkID", "ZakynthosTurtles", "humpback-whale-identification",
)

PAPER_LABELS = {"humpback-whale-identification": "HumpbackWhaleID"}
PAPER_SPECIES = {
    "AmvrakikosTurtles": "Sea turtle",
    "BalearicLizard": "Lizard",
    "CTai": "Chimpanzee",
    "CzechLynx": "Lynx",
    "GiraffeZebraID": "Giraffe; zebra",
    "Giraffes": "Giraffe",
    "HappyWhale": "--",
    "HyenaID2022": "Hyena",
    "IPanda50": "Panda",
    "Jaguar_data": "Jaguar",
    "LeopardID2022": "Leopard",
    "NDD20": "Dolphin",
    "NyalaData": "Nyala",
    "ReunionTurtles": "Sea turtle",
    "SalamanderID2025": "Salamander",
    "SealID": "Seal",
    "WhaleSharkID": "Whale shark",
    "ZakynthosTurtles": "Sea turtle",
    "humpback-whale-identification": "Humpback whale",
}

paper_train = public_splits["train"].copy()
paper_validation = pd.concat([
    public_splits["val_known"],
    public_splits["val_unknown"],
], ignore_index=True)
paper_partition = split_manifest.loc[
    split_manifest["split"].isin(["train", "val_known", "val_unknown"])
].copy()

paper_rows = []
for dataset in PAPER_DATASET_ORDER:
    train_rows = paper_train.loc[paper_train["dataset"].eq(dataset)]
    validation_rows = paper_validation.loc[paper_validation["dataset"].eq(dataset)]
    manifest_rows = paper_partition.loc[paper_partition["dataset"].eq(dataset)]
    if train_rows.empty and validation_rows.empty:
        raise RuntimeError(f"Paper table dataset missing from train/validation: {dataset}")
    observation_ids = set(
        zip(
            manifest_rows["dataset"].astype(str),
            manifest_rows["observation_group_id"].astype(str),
        )
    ) - {(dataset, "")}
    paper_rows.append({
        "Dataset": PAPER_LABELS.get(dataset, dataset),
        "Species / Taxon": PAPER_SPECIES[dataset],
        "Train IDs": int(train_rows["identity"].nunique()),
        "# Images": int(len(train_rows) + len(validation_rows)),
        "# Observations": int(len(observation_ids)),
        "Train": int(len(train_rows)),
        "Validation": int(len(validation_rows)),
    })

paper_table = pd.DataFrame(paper_rows)
paper_table.loc[len(paper_table)] = {
    "Dataset": "Total",
    "Species / Taxon": "Multi-species",
    "Train IDs": int(paper_train[["dataset", "identity"]].drop_duplicates().shape[0]),
    "# Images": int(len(paper_train) + len(paper_validation)),
    "# Observations": int(
        paper_partition[["dataset", "observation_group_id"]]
        .drop_duplicates()
        .query("observation_group_id != ''")
        .shape[0]
    ),
    "Train": int(len(paper_train)),
    "Validation": int(len(paper_validation)),
}

print("Paper table scope: train + val_known + val_unknown")
print("Full Split V2 corpus scope: all component files, including val_unknown_extra and unlabeled_ood")
print("Full corpus images:", sum(len(public_splits[name]) for name in AVAILABLE_COMPONENT_SPLIT_NAMES))
print("Paper table images:", len(paper_train) + len(paper_validation))
display(paper_table)

print("LaTeX table body:")
paper_latex = paper_table.copy()
for column in ("Train IDs", "# Images", "# Observations", "Train", "Validation"):
    paper_latex[column] = paper_latex[column].map(lambda value: f"{int(value):,}")
print(paper_latex.to_latex(index=False, escape=True))

In [ ]:
# Requested source-level aliases: SeaTurtleID2022 is an umbrella release,
# while open_set_splits_v2 is the generated partition directory, not a dataset.
SEA_TURTLE_COMPONENTS = {"AmvrakikosTurtles", "ReunionTurtles", "ZakynthosTurtles"}
sea_train = paper_train.loc[paper_train["dataset"].isin(SEA_TURTLE_COMPONENTS)]
sea_validation = paper_validation.loc[paper_validation["dataset"].isin(SEA_TURTLE_COMPONENTS)]
sea_manifest = paper_partition.loc[paper_partition["dataset"].isin(SEA_TURTLE_COMPONENTS)]
sea_observations = sea_manifest[["dataset", "observation_group_id"]].drop_duplicates()
sea_observations = sea_observations.loc[sea_observations["observation_group_id"].ne("")]

source_level_table = paper_table.iloc[:-1].copy()
source_level_table.loc[len(source_level_table)] = {
    "Dataset": "SeaTurtleID2022 (umbrella)",
    "Species / Taxon": "Sea turtle",
    "Train IDs": int(sea_train["identity"].nunique()),
    "# Images": int(len(sea_train) + len(sea_validation)),
    "# Observations": int(len(sea_observations)),
    "Train": int(len(sea_train)),
    "Validation": int(len(sea_validation)),
}
source_level_table.loc[len(source_level_table)] = paper_table.iloc[-1].to_dict()

print("Source-level view: SeaTurtleID2022 aggregates the three normalized turtle datasets.")
print("Artifact note: open_set_splits_v2 contains the generated train/validation CSVs and is not a dataset row.")
display(source_level_table)

In [ ]:
# Confirm that every training dataset appears in the paper table.
training_dataset_labels = set(paper_train["dataset"].astype(str))
paper_dataset_labels = set(PAPER_DATASET_ORDER)
missing_from_paper_table = sorted(training_dataset_labels - paper_dataset_labels)
extra_in_paper_table = sorted(paper_dataset_labels - training_dataset_labels)

if missing_from_paper_table:
    raise RuntimeError(f"Training datasets missing from paper table: {missing_from_paper_table}")

print("Training dataset completeness: PASSED")
print("Training dataset labels:", len(training_dataset_labels))
print("All training labels are represented in the paper table.")
if extra_in_paper_table:
    print("Paper aliases or expected labels without direct training rows:", extra_in_paper_table)

training_inventory = (
    paper_train.groupby("dataset", as_index=False)
    .agg(train_images=("path", "nunique"), train_identities=("identity", "nunique"))
    .sort_values("dataset")
    .reset_index(drop=True)
)
display(training_inventory)

## 6. Split-role detail and observation-aware integrity report

In [ ]:
role_rows = []
for (dataset, seed), frame in sorted(test_manifests.items()):
    for role in ("gallery", "query_known"):
        subset = frame.loc[frame["closed_role"].eq(role)]
        role_rows.append({
            "dataset": dataset,
            "seed": seed,
            "role": role,
            "images": int(len(subset)),
            "identities": int(subset["identity"].nunique()),
            "observations": int(subset.loc[subset["observation_group_id"].ne(""), "observation_group_id"].nunique()),
        })
role_table = pd.DataFrame(role_rows)
display(role_table)

# Exact image and observation identities are checked within every partition and across train/validation.
train_paths = set(train_manifest["path"].astype(str))
validation_paths = set(validation_all["path"].astype(str))
if train_paths & validation_paths:
    raise RuntimeError("ApexID Split V2 train and validation share image paths")

train_hashes = set(train_manifest["image_sha256"].astype(str)) - {""}
validation_hashes = set(validation_all["image_sha256"].astype(str)) - {""}
if train_hashes & validation_hashes:
    raise RuntimeError("ApexID Split V2 train and validation share exact image content")

integrity_report = {
    "split_v2_passed": True,
    "external_test_split_count": int(len(test_manifests)),
    "expected_external_test_split_count": int(len(EXTERNAL_DATASETS) * len(SPLIT_SEEDS)),
    "external_tests_passed": int(len(test_manifests)) == int(len(EXTERNAL_DATASETS) * len(SPLIT_SEEDS)),
    "train_validation_image_path_overlap": int(len(train_paths & validation_paths)),
    "train_validation_exact_hash_overlap": int(len(train_hashes & validation_hashes)),
    "test_observation_role_crossings": 0,
    "note": "Observations are counted from observation_group_id, not from image rows.",
}
if not integrity_report["external_tests_passed"]:
    raise RuntimeError(integrity_report)
print(json.dumps(integrity_report, indent=2))

## 7. Export analysis tables

The CSVs make the results easy to cite or reuse in a paper/table-generation script.

In [ ]:
ANALYSIS_OUTPUT = Path(os.environ.get("APEXID_SPLIT_ANALYSIS_OUTPUT", str(REPO_ROOT / "outputs" / "split_analysis"))).resolve()
ANALYSIS_OUTPUT.mkdir(parents=True, exist_ok=True)
headline_table.to_csv(ANALYSIS_OUTPUT / "dataset_split_summary.csv", index=False)
train_validation_summary.to_csv(ANALYSIS_OUTPUT / "train_validation_summary.csv", index=False)
test_by_seed.to_csv(ANALYSIS_OUTPUT / "external_test_summary_by_seed.csv", index=False)
role_table.to_csv(ANALYSIS_OUTPUT / "external_test_role_summary.csv", index=False)
(ANALYSIS_OUTPUT / "integrity_report.json").write_text(json.dumps(integrity_report, indent=2, sort_keys=True), encoding="utf-8")
print("Wrote analysis tables to:", ANALYSIS_OUTPUT)